## Step1: the condition-to-guideline mapping

In [ ]:
# list the guidelines in the KB for each specialty (the number of pages, the CCSR code, the pdf names) .
import json, collections
from pathlib import Path

KB_DIR = Path("Knowledge Base/Processed_v2")

for f in sorted(KB_DIR.glob("*.jsonl")):
    counts = collections.Counter()
    with f.open(encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                counts[json.loads(line).get("source_file")] += 1
    print(f"\n=== {f.name}: {sum(counts.values()):,} chunks, {len(counts)} guidelines")
    for src, n in sorted(counts.items()):
        print(f"  {n:>5}  {src}")

In [ ]:
# Confirm the CCSR descriptions of the 19 codes.
import pandas as pd

TARGETS = ["CIR017","CIR019","CIR008","CIR011","CIR007",
           "RSP012","RSP002","RSP008","RSP011","RSP010",
           "GEN002","END011","GEN003","GEN006","GEN001",
           "INF003","INF002","GEN004","INF004"]

xl = pd.ExcelFile("DXCCSR-Reference-File-v2026-1.xlsx")
print("Sheets:", xl.sheet_names)

found = {}
for sheet in xl.sheet_names:
    df = xl.parse(sheet, dtype=str, header=None)
    if len(df) > 2000:          # skip the large ICD-to-CCSR mapping sheet
        continue
    for _, row in df.iterrows():
        cells = [str(c).strip() for c in row if pd.notna(c)]
        for code in TARGETS:
            if code in cells and code not in found:
                found[code] = " | ".join(cells[:3])

for code in TARGETS:
    print(f"{code}: {found.get(code, 'NOT FOUND')}")

In [ ]:
# Build, check and save condition_guideline_map.json
import json, collections
from pathlib import Path

KB_DIR = Path("Knowledge Base/Processed_v2")
FOLDERS = {"cardiology": "cardiology_chunks.jsonl",
           "infectious_diseases": "infectious_diseases_chunks.jsonl",
           "nephrology": "nephrology_chunks.jsonl",
           "pulmonology": "pulmonology_chunks.jsonl"}
EXCLUDE = ["Data-Supplement"]          # GEN002 data supplement: not mapped

# chunk counts per (folder, source_file)
kb_files = {}
for folder, fname in FOLDERS.items():
    c = collections.Counter()
    with (KB_DIR / fname).open(encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                c[json.loads(line)["source_file"]] += 1
    kb_files[folder] = c

# (specialist, code, name, folder to search, label key; None = whole folder, rule note)
SLOTS = [
 ("Cardiology","CIR017","Cardiac dysrhythmias","cardiology","CIR017","label"),
 ("Cardiology","CIR019","Heart failure","cardiology","CIR019","label"),
 ("Cardiology","CIR008","Hypertension with complications and secondary hypertension","cardiology","CIR008 and CIR007","label (shared with CIR007)"),
 ("Cardiology","CIR011","Coronary atherosclerosis and other heart disease","cardiology","CIR011","label"),
 ("Cardiology","CIR007","Essential hypertension","cardiology","CIR008 and CIR007","label (shared with CIR008)"),
 ("Pulmonology","RSP012","Respiratory failure; insufficiency; arrest","pulmonology","RSP012","label"),
 ("Pulmonology","RSP002","Pneumonia (except that caused by tuberculosis)","pulmonology","RSP002","label"),
 ("Pulmonology","RSP008","Chronic obstructive pulmonary disease and bronchiectasis","pulmonology","RSP008","label"),
 ("Pulmonology","RSP011","Pleurisy, pleural effusion and pulmonary collapse","pulmonology","RSP011","label"),
 ("Pulmonology","RSP010","Aspiration pneumonitis","pulmonology","RSP010","label"),
 ("Nephrology","GEN002","Acute and unspecified renal failure","nephrology","GEN002","label"),
 ("Nephrology","END011","Fluid and electrolyte disorders","nephrology","GEN003","exception: covered by KDIGO 2024 CKD"),
 ("Nephrology","GEN003","Chronic kidney disease","nephrology","GEN003","label"),
 ("Nephrology","GEN006","Other specified and unspecified diseases of kidney and ureters","nephrology",None,"fallback: no specific guideline -> whole specialist folder"),
 ("Nephrology","GEN001","Nephritis; nephrosis; renal sclerosis","nephrology","GEN001","label"),
 ("Infectious Diseases","INF003","Bacterial infections","infectious_diseases","INF003","label"),
 ("Infectious Diseases","INF002","Septicemia","infectious_diseases","INF002","label"),
 ("Infectious Diseases","RSP002","Pneumonia (except that caused by tuberculosis)","pulmonology","RSP002","exception: cross-folder (BTS CAP)"),
 ("Infectious Diseases","GEN004","Urinary tract infections","nephrology","GEN006","exception: UTI files carry GEN006 label"),
 ("Infectious Diseases","INF004","Fungal infections","infectious_diseases","INF004","label"),
]

cmap = {}
for spec, code, name, folder, key, rule in SLOTS:
    files = [s for s in kb_files[folder]
             if (key is None or key in s) and not any(x in s for x in EXCLUDE)]
    assert files, f"No guideline files for {spec}/{code}"
    n = sum(kb_files[folder][s] for s in files)
    cmap.setdefault(spec, {})[code] = {
        "name": name, "rule": rule, "kb_folder": folder,
        "source_files": sorted(files), "n_chunks": n}
    print(f"{spec:<20} {code}  files={len(files):>2}  chunks={n:>5}  [{rule}]")

with open("condition_guideline_map.json", "w", encoding="utf-8") as f:
    json.dump(cmap, f, indent=2, ensure_ascii=False)
print("\nSaved condition_guideline_map.json |",
      sum(len(v) for v in cmap.values()), "slots")

## Step 2: patient retrieval queries

In [ ]:
import json

PATIENT_QUERIES = {
 # ---------------- Cardiology (unchanged from 5-Patient_Case_RAG cell 5)
 "CIR017": ("Evidence relevant to cardiac dysrhythmias, including the patient's "
            "cardiac rhythm, rate, conduction, electrocardiographic findings, "
            "rhythm-related symptoms, cardiac history, and rhythm-related "
            "treatments or procedures."),
 "CIR019": ("Evidence relevant to heart failure, including cardiac function, "
            "volume status, symptoms and signs of cardiac congestion, "
            "cardiopulmonary findings, cardiac investigations, and treatments "
            "related to cardiac function or volume management."),
 "CIR008": ("Evidence relevant to hypertension with complications or secondary "
            "hypertension, including blood pressure abnormalities, possible "
            "secondary causes, associated organ dysfunction, cardiovascular or "
            "renal complications, and antihypertensive management."),
 "CIR011": ("Evidence relevant to coronary atherosclerosis and other ischemic "
            "heart disease, including ischemic symptoms, cardiac biomarkers, "
            "electrocardiographic findings, cardiac investigations, coronary "
            "history, and treatments related to myocardial ischemia."),
 "CIR007": ("Evidence relevant to essential hypertension, including blood pressure "
            "measurements and trends, history of hypertension, cardiovascular "
            "assessment, and antihypertensive treatment."),
 # ---------------- Pulmonology
 "RSP012": ("Evidence relevant to respiratory failure or insufficiency, including "
            "oxygenation and ventilation status, arterial blood gas findings, "
            "respiratory rate and work of breathing, supplemental oxygen, "
            "non-invasive or mechanical ventilation, intubation, and respiratory arrest."),
 "RSP002": ("Evidence relevant to pneumonia, including cough, sputum, fever, "
            "chest auscultation findings, chest imaging showing consolidation or "
            "infiltrates, inflammatory markers, respiratory cultures, and "
            "antibiotic treatment for respiratory infection."),
 "RSP008": ("Evidence relevant to chronic obstructive pulmonary disease or "
            "bronchiectasis, including chronic lung disease history, smoking history, "
            "wheeze or airflow obstruction, exacerbation symptoms, pulmonary function, "
            "chronic sputum production, and inhaled bronchodilators or corticosteroids."),
 "RSP011": ("Evidence relevant to pleural effusion, pleurisy or pulmonary collapse, "
            "including pleuritic chest pain, reduced breath sounds, chest imaging "
            "showing effusion, atelectasis or lung collapse, thoracentesis, "
            "pleural fluid analysis, and chest tubes."),
 "RSP010": ("Evidence relevant to aspiration pneumonitis, including witnessed "
            "aspiration or vomiting, dysphagia or swallowing evaluation, reduced "
            "consciousness or impaired airway protection, feeding tubes, and chest "
            "imaging showing dependent or lower-lobe opacities."),
 # ---------------- Nephrology
 "GEN002": ("Evidence relevant to acute kidney injury or renal failure, including "
            "creatinine and blood urea nitrogen trends, urine output or oliguria, "
            "nephrotoxic exposures, renal imaging, fluid management, and renal "
            "replacement therapy or dialysis."),
 "END011": ("Evidence relevant to fluid and electrolyte disorders, including sodium, "
            "potassium, chloride, bicarbonate, calcium, magnesium and phosphate "
            "abnormalities, acid-base status, volume depletion or overload, and "
            "electrolyte replacement or correction."),
 "GEN003": ("Evidence relevant to chronic kidney disease, including history of "
            "chronic kidney disease, baseline renal function, estimated glomerular "
            "filtration rate, proteinuria or albuminuria, prior dialysis, anemia of "
            "kidney disease, and long-term kidney-related medications."),
 "GEN006": ("Evidence relevant to other diseases of the kidney and ureter, including "
            "hydronephrosis, ureteral obstruction or stents, renal cysts or masses, "
            "renal vascular disease, abnormal renal or urological imaging, "
            "nephrostomy, and urological procedures."),
 "GEN001": ("Evidence relevant to glomerular disease, nephritis, nephrotic syndrome "
            "or renal sclerosis, including proteinuria, hematuria, edema, low albumin, "
            "urinalysis findings, kidney biopsy, hypertensive or diabetic kidney "
            "disease, and immunosuppressive treatment."),
 # ---------------- Infectious Diseases (RSP002 shared above)
 "INF003": ("Evidence relevant to bacterial infection, including fever, white blood "
            "cell count and inflammatory markers, blood, urine, sputum or wound "
            "cultures, identified bacterial organisms and susceptibilities, "
            "infection source, and antibiotic therapy."),
 "INF002": ("Evidence relevant to sepsis or septicemia, including fever or "
            "hypothermia, tachycardia, hypotension, lactate, blood cultures and "
            "bacteremia, organ dysfunction, fluid resuscitation, vasopressor support, "
            "and broad-spectrum antibiotics."),
 "GEN004": ("Evidence relevant to urinary tract infection, including urinary "
            "symptoms, urinalysis with leukocytes or nitrites, urine culture, "
            "urinary catheter use, pyelonephritis or urosepsis, and antibiotic "
            "treatment for urinary infection."),
 "INF004": ("Evidence relevant to fungal infection, including Candida or other fungal "
            "organisms in cultures, oral or esophageal candidiasis, fungemia, risk "
            "factors such as central lines, broad-spectrum antibiotics or "
            "immunosuppression, and antifungal therapy."),
}

# check: every condition in the frozen map has exactly one query
cmap = json.load(open("condition_guideline_map.json"))
map_codes = {c for spec in cmap.values() for c in spec}
assert map_codes == set(PATIENT_QUERIES), (map_codes ^ set(PATIENT_QUERIES))

with open("patient_queries.json", "w", encoding="utf-8") as f:
    json.dump(PATIENT_QUERIES, f, indent=2)
print("Saved patient_queries.json |", len(PATIENT_QUERIES), "queries")

In [ ]:
#To make sure the GEN006 query covers what is actually in that family, this lists the ICD-10 code descriptions that CCSR maps to GEN006.

import pandas as pd
m = pd.read_excel("DXCCSR-Reference-File-v2026-1.xlsx", sheet_name="DX_to_CCSR_Mapping",
                  dtype=str, header=None)
rows = m[m.apply(lambda r: r.astype(str).str.contains("GEN006").any(), axis=1)]
print(len(rows), "ICD-10 codes map to GEN006")
print(rows.iloc[:, :2].to_string(index=False, header=False))

Tlist shows the query needs changing. It covers obstruction well, but it misses the renal tubular group, and that group is probably the most common in your cohort. N25.81, secondary hyperparathyroidism of renal origin, is a frequent code in ICU patients with CKD. The query also misses small or atrophic kidney and renal infarction, and it wrongly includes "renal masses" (those are coded as neoplasms, not GEN006). For that the related query needs correction:
Revised GEN006 query, covering the four main groups in your list: obstruction, tubular/mineral-bone disorders, structural changes, and vascular causes.

In [ ]:
#Revised GEN006 query, covering the four main groups in your list: obstruction, tubular/mineral-bone disorders, structural changes, and vascular causes.

import json

q = json.load(open("patient_queries.json"))
q["GEN006"] = ("Evidence relevant to other disorders of the kidney and ureter, including "
               "hydronephrosis, hydroureter or obstructive uropathy, ureteral stricture "
               "or stents, renal colic, secondary hyperparathyroidism of renal origin, "
               "renal osteodystrophy, impaired renal tubular function, small or atrophic "
               "kidneys, renal cysts, renal ischemia or infarction, and renal or "
               "urological imaging findings.")
with open("patient_queries.json", "w", encoding="utf-8") as f:
    json.dump(q, f, indent=2)
print("Updated GEN006 |", len(q), "queries")

## Step 3: patient retrieval

This cell retrieves the top 10 chunks for each of the 60 cases and each of the 19 conditions, searching only that patient's admission. It saves row indices and scores but no text, so the file stays small; the text is attached later, in step 5.

In [ ]:
import json, numpy as np, pandas as pd, collections
from sentence_transformers import SentenceTransformer

TOP_K_PATIENT = 10

# ---- inputs
cases   = pd.read_csv("cases_60.csv", dtype=str)
queries = json.load(open("patient_queries.json"))
chunks  = [json.loads(l) for l in open("all_patient_chunks_v2.jsonl", encoding="utf-8") if l.strip()]
P_EMB   = np.load("patient_chunk_embeddings_v2.npy")
assert len(chunks) == len(P_EMB), "chunk/embedding misalignment"

norms = np.linalg.norm(P_EMB, axis=1)
print(f"Patient embedding norms: min={norms.min():.4f} max={norms.max():.4f}")  # expect ~1.0

# ---- row indices per admission
rows_by_hadm = collections.defaultdict(list)
for i, c in enumerate(chunks):
    rows_by_hadm[str(c["hadm_id"])].append(i)

# ---- encode the 19 queries once (no prefix, as in smoke test v2)
bge = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cpu")
codes = list(queries)
Q_EMB = bge.encode([queries[c] for c in codes], convert_to_numpy=True,
                   normalize_embeddings=True)
Q = dict(zip(codes, Q_EMB))

# ---- retrieve
patient_retrieval = {}
short_admissions = 0
for hadm in cases["hadm_id"]:
    rows = rows_by_hadm[hadm]
    assert rows, f"no chunks for {hadm}"
    if len(rows) < TOP_K_PATIENT:
        short_admissions += 1
    sims_all = P_EMB[rows] @ np.stack([Q[c] for c in codes]).T   # (n_rows, 19)
    per_cond = {}
    for j, code in enumerate(codes):
        order = np.argsort(sims_all[:, j])[::-1][:TOP_K_PATIENT]
        per_cond[code] = [{"row": int(rows[k]),
                           "chunk_id": chunks[rows[k]].get("chunk_id"),
                           "section": chunks[rows[k]]["section"],
                           "score": round(float(sims_all[k, j]), 4)} for k in order]
    patient_retrieval[hadm] = per_cond

with open("patient_retrieval.json", "w", encoding="utf-8") as f:
    json.dump({"top_k": TOP_K_PATIENT, "query_prefix": None,
               "embedding_model": "BAAI/bge-base-en-v1.5",
               "results": patient_retrieval}, f)

# ---- summary (counts only)
uniq = [len({r["row"] for cond in v.values() for r in cond}) for v in patient_retrieval.values()]
scores = [r["score"] for v in patient_retrieval.values() for cond in v.values() for r in cond]
sec = collections.Counter(r["section"] for v in patient_retrieval.values()
                          for cond in v.values() for r in cond)
print(f"Cases: {len(patient_retrieval)} | conditions per case: {len(codes)}")
print(f"Admissions with < {TOP_K_PATIENT} chunks: {short_admissions}")
print(f"Unique chunks per case after dedup: min={min(uniq)} median={int(np.median(uniq))} max={max(uniq)}  (max possible {len(codes)*TOP_K_PATIENT})")
print(f"Similarity scores: min={min(scores):.3f} median={np.median(scores):.3f} max={max(scores):.3f}")
print("Retrieved slots by section:")
for s, n in sec.most_common():
    print(f"  {n:>6}  {s}")

In [ ]:
# section breakdown per condition (counts only)

import json, collections

pr = json.load(open("patient_retrieval.json"))["results"]
codes = list(next(iter(pr.values())).keys())

print(f"{'Code':<8}{'top sections (% of 600 slots)'}")
for code in codes:
    c = collections.Counter(r["section"] for case in pr.values() for r in case[code])
    total = sum(c.values())
    top = ", ".join(f"{s.title()[:18]} {100*n/total:.0f}%" for s, n in c.most_common(4))
    print(f"{code:<8}{top}")

# how many cases get at least one chunk from key sections, for the conditions that need them
checks = {"INF003": "MICROBIOLOGY", "INF002": "MICROBIOLOGY", "GEN004": "MICROBIOLOGY",
          "INF004": "MICROBIOLOGY", "CIR017": "PATIENT ECG", "RSP012": "BLOOD GAS",
          "END011": "PATIENT CHEMISTRY", "GEN002": "LABORATORY RESULTS"}
print("\nCases (of 60) where the condition retrieved >= 1 chunk from its key section:")
for code, sec in checks.items():
    n = sum(any(r["section"] == sec for r in case[code]) for case in pr.values())
    print(f"  {code:<8}{sec:<22}{n:>3} / 60")

In [ ]:
# This cell measures both things in one run: the real denominators, and what the cap would change. 
# It uses the variables still loaded from cell 4. If you restarted the kernel, run cell 4 again first.
import numpy as np, collections

CAP = 5
checks = {"INF003": "MICROBIOLOGY", "INF002": "MICROBIOLOGY", "GEN004": "MICROBIOLOGY",
          "INF004": "MICROBIOLOGY", "CIR017": "PATIENT ECG", "RSP012": "BLOOD GAS",
          "END011": "PATIENT CHEMISTRY"}

def topk(rows, sims, cap=None, k=10):
    out, per_sec = [], collections.Counter()
    for i in np.argsort(sims)[::-1]:
        s = chunks[rows[i]]["section"]
        if cap and per_sec[s] >= cap:
            continue
        out.append(rows[i]); per_sec[s] += 1
        if len(out) == k:
            break
    return out

has_sec = collections.Counter()
hit = {"nocap": collections.Counter(), "cap": collections.Counter()}
radiology_share = {"nocap": [], "cap": []}
uniq = {"nocap": [], "cap": []}

for hadm in cases["hadm_id"]:
    rows = rows_by_hadm[hadm]
    secs_present = {chunks[r]["section"] for r in rows}
    for sec in set(checks.values()):
        has_sec[sec] += sec in secs_present
    sims_all = P_EMB[rows] @ np.stack([Q[c] for c in codes]).T
    for mode in ("nocap", "cap"):
        all_rows = set()
        for j, code in enumerate(codes):
            got = topk(rows, sims_all[:, j], cap=CAP if mode == "cap" else None)
            all_rows.update(got)
            radiology_share[mode].append(sum(chunks[r]["section"] == "RADIOLOGY REPORTS" for r in got) / 10)
            if code in checks and any(chunks[r]["section"] == checks[code] for r in got):
                hit[mode][code] += 1
        uniq[mode].append(len(all_rows))

print(f"{'Code':<8}{'Section':<20}{'cases having it':>16}{'hit no cap':>12}{'hit cap 5':>11}")
for code, sec in checks.items():
    print(f"{code:<8}{sec:<20}{has_sec[sec]:>16}{hit['nocap'][code]:>12}{hit['cap'][code]:>11}")
for mode in ("nocap", "cap"):
    print(f"\n{mode}: radiology share {100*np.mean(radiology_share[mode]):.0f}% | "
          f"unique chunks/case median {int(np.median(uniq[mode]))} (max {max(uniq[mode])})")

## Step 4: summary queries (vLLM)

#### First sub-step: start vLLM the same way done for the Model 1 run, then check that it answers.
#### Step 4b: summary queries

How the cell works:

* For each case and each of the 19 conditions, it sends that condition's 10 retrieved patient chunks to MedGemma 27B with your old cell 14 prompt (no numbers, no diagnosis, one paragraph), plus a 200-word limit so the summary fits BGE's 512-token input.
* Settings: temperature 0, a 300-token output cap, and 4 requests in parallel so vLLM can batch them.
* It saves after every call and skips anything already done, so if it stops it continues where it left off.
* It prints only counts and timing. The summaries contain patient-derived content, so view them in Workbench and don't paste them here.

In [ ]:
import json, time, requests, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

RUN_MODE   = "full"          # "test" = 1 case (19 calls); then change to "full"
URL        = "http://127.0.0.1:8000/v1/chat/completions"
MODEL      = "bbarn4/medgemma-27b-text-it-GPTQ"
MAX_TOKENS = 300
WORKERS    = 4
OUT        = Path("kb_query_summaries.json")

cmap    = json.load(open("condition_guideline_map.json"))
pr      = json.load(open("patient_retrieval.json"))["results"]
chunks  = [json.loads(l) for l in open("all_patient_chunks_v2.jsonl", encoding="utf-8") if l.strip()]
names   = {code: v["name"] for spec in cmap.values() for code, v in spec.items()}

PROMPT = """You are creating a concise clinical semantic summary for
medical knowledge-base retrieval.

Candidate condition: {name} ({code})

IMPORTANT:
The candidate condition is a retrieval hypothesis only.
Do NOT assume that the patient has this condition.
Do NOT make a diagnosis.

Using ONLY the patient evidence provided below, summarize the
clinical concepts relevant to assessing or managing the candidate
condition.

Rules:
- Include only concepts explicitly supported by the patient evidence.
- Focus on clinically meaningful concepts useful for guideline retrieval.
- Include relevant symptoms and signs.
- Include relevant laboratory concepts without numerical values.
- Include relevant imaging findings.
- Include relevant comorbidities and organ dysfunction.
- Include relevant documented treatments or therapies.
- Include competing conditions when they may affect interpretation.
- Do NOT include patient-specific numerical values.
- Do NOT include age or other demographic details.
- Do NOT invent or infer findings.
- Do NOT generate new recommendations.
- Do NOT convert planned/recommended management into completed treatment.
- Do NOT include generic categories such as "labs", "symptoms",
  "medical history", or "imaging findings".
- Avoid repetition.
- Do NOT cite, number or refer to the evidence sources or sections.
- Write one concise clinical paragraph of no more than 200 words.
- The summary will be used only as a semantic query for
  knowledge-base retrieval.
- Describe documented findings and conditions without deciding whether
  the candidate condition is currently present, absent, acute,
  decompensated, exacerbated, or otherwise diagnostically classified.

Return ONLY the clinical semantic summary.

PATIENT EVIDENCE:
{evidence}"""

def build_prompt(hadm, code):
    blocks = [f"Section: {chunks[r['row']]['section']}\nText: {chunks[r['row']]['text']}"
              for r in pr[hadm][code]]
    return PROMPT.format(name=names[code], code=code, evidence="\n\n".join(blocks))

done = json.load(open(OUT)) if OUT.exists() else {}
lock = threading.Lock()

hadms = list(pr)[:1] if RUN_MODE == "test" else list(pr)
codes = list(names)
todo  = [(h, c) for h in hadms for c in codes if f"{h}|{c}" not in done]
print(f"Mode={RUN_MODE} | to run: {len(todo)} | already done: {len(done)}")

def call(h, c):
    body = {"model": MODEL, "temperature": 0, "max_tokens": MAX_TOKENS,
            "messages": [{"role": "user", "content": build_prompt(h, c)}]}
    r = requests.post(URL, json=body, timeout=600); r.raise_for_status()
    j = r.json()
    return h, c, {"summary": j["choices"][0]["message"]["content"].strip(),
                  "finish": j["choices"][0]["finish_reason"],
                  "prompt_tokens": j["usage"]["prompt_tokens"],
                  "completion_tokens": j["usage"]["completion_tokens"]}

t0, errors = time.time(), 0
with ThreadPoolExecutor(WORKERS) as ex:
    futs = [ex.submit(call, h, c) for h, c in todo]
    for n, f in enumerate(as_completed(futs), 1):
        try:
            h, c, res = f.result()
            with lock:
                done[f"{h}|{c}"] = res
                OUT.write_text(json.dumps(done))
        except Exception as e:
            errors += 1; print("Error:", repr(e)[:200])
        if n % 19 == 0 or n == len(todo):
            el = time.time() - t0
            print(f"  {n}/{len(todo)} done | {el/60:.1f} min | est. total {el/n*len(todo)/60:.1f} min")

# summary (counts only)
sel = [done[f"{h}|{c}"] for h in hadms for c in codes if f"{h}|{c}" in done]
print(f"\nErrors: {errors} | stored: {len(sel)}")
print(f"Hit token cap (finish=length): {sum(s['finish']=='length' for s in sel)}")
print(f"Prompt tokens:  max {max(s['prompt_tokens'] for s in sel):,}")
print(f"Output tokens:  min {min(s['completion_tokens'] for s in sel)} | max {max(s['completion_tokens'] for s in sel)}")
print(f"Time per call (wall): {(time.time()-t0)/max(1,len(todo)):.1f} s")

In [ ]:
for k, v in done.items():
    if v["finish"] == "length":
        print("=" * 80, "\n", k.split("|")[1], "\n", v["summary"][-600:])

In [ ]:
import json, re
d = json.load(open("kb_query_summaries.json"))
s = [v["summary"] for v in d.values()]
print("Summaries:", len(s))
print("Mention 'chunk':        ", sum("chunk" in x.lower() for x in s))
print("Contain digits:         ", sum(bool(re.search(r"\d", x)) for x in s))
print("Multi-line / bulleted:  ", sum(("\n" in x.strip()) or x.lstrip().startswith(("-", "*", "•")) for x in s))
print("Words: min", min(len(x.split()) for x in s), "| max", max(len(x.split()) for x in s))

In [ ]:
# Check what the digits are in the above chunks. This shows each number with a few words around it
for k, v in d.items():
    hits = re.findall(r"(?:\S+\s){0,3}\S*\d\S*(?:\s\S+){0,2}", v["summary"])
    if hits:
        print(k.split("|")[1], "->", hits)

In [ ]:
import json, time, collections, numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

RUN_MODE = "full"            # "test" = cases with all 19 summaries; "full" = all 60
TOP_K_KB, FLOOR = 10, 5
KB_DIR, EMB_DIR = Path("Knowledge Base/Processed_v2"), Path("Knowledge Base/Embeddings")
SPEC_FOLDER = {"Cardiology": "cardiology", "Pulmonology": "pulmonology",
               "Nephrology": "nephrology", "Infectious Diseases": "infectious_diseases"}
EXCLUDE = ["Data-Supplement"]

cmap = json.load(open("condition_guideline_map.json"))

# ---- load KB chunks + embeddings, aligned by chunk_id
kb_chunks, kb_emb, kb_folder = [], [], []
for folder in SPEC_FOLDER.values():
    ch = [json.loads(l) for l in open(KB_DIR / f"{folder}_chunks.jsonl", encoding="utf-8") if l.strip()]
    z = np.load(EMB_DIR / f"{folder}_bge-base-en-v1.5.npz", allow_pickle=True)
    id_key  = next(k for k in z.files if "id" in k.lower())
    emb_key = next(k for k in z.files if "emb" in k.lower())
    ids, embs = z[id_key], z[emb_key]          # read each array ONCE
    pos = {str(i): k for k, i in enumerate(ids)}
    assert len(pos) == len(ids), f"duplicate ids in {folder}"
    for c in ch:
        kb_chunks.append(c); kb_emb.append(embs[pos[str(c["chunk_id"])]]); kb_folder.append(folder)
E = np.vstack(kb_emb)
n = np.linalg.norm(E, axis=1)
print(f"KB chunks: {len(kb_chunks):,} | norms {n.min():.4f}-{n.max():.4f}")

by_file = collections.defaultdict(list)
for i, c in enumerate(kb_chunks):
    by_file[(kb_folder[i], c["source_file"])].append(i)

# ---- mapped sets and search pools
mapped = {spec: {code: {i for f in v["source_files"] for i in by_file[(v["kb_folder"], f)]}
                 for code, v in conds.items()} for spec, conds in cmap.items()}
pool = {}
for spec, folder in SPEC_FOLDER.items():
    p = {i for i in range(len(kb_chunks))
         if kb_folder[i] == folder and not any(x in kb_chunks[i]["source_file"] for x in EXCLUDE)}
    for code in mapped[spec]:
        p |= mapped[spec][code]           # adds BTS CAP + UTI files for ID
    pool[spec] = np.array(sorted(p))
    for code, m in mapped[spec].items():
        assert m <= set(pool[spec]), f"{spec}/{code} mapped files outside pool"
print("Pool sizes:", {s: len(p) for s, p in pool.items()})

# ---- summaries (file is being written by the other notebook: retry on partial read)
for _ in range(5):
    try:
        summ = json.load(open("kb_query_summaries.json")); break
    except json.JSONDecodeError:
        time.sleep(2)
codes = sorted({c for conds in cmap.values() for c in conds})
hadms = sorted({k.split("|")[0] for k in summ})
hadms = [h for h in hadms if all(f"{h}|{c}" in summ for c in codes)]
if RUN_MODE == "test":
    hadms = hadms[:1]
print(f"Cases with complete summaries used: {len(hadms)}")

bge = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cpu")
keys = [f"{h}|{c}" for h in hadms for c in codes]
QV = dict(zip(keys, bge.encode([summ[k]["summary"] for k in keys], batch_size=32,
                               convert_to_numpy=True, normalize_embeddings=True)))

def retrieve(qv, pool_idx, mset):
    sims = E[pool_idx] @ qv
    ranked = pool_idx[np.argsort(sims)[::-1]]
    score = dict(zip(pool_idx, sims))
    top = list(ranked[:TOP_K_KB])
    n_m = sum(i in mset for i in top)
    swaps = 0
    if n_m < FLOOR:
        extra = [i for i in ranked[TOP_K_KB:] if i in mset][:FLOOR - n_m]
        drop = set([i for i in top if i not in mset][-len(extra):])
        top = [i for i in top if i not in drop] + extra
        swaps = len(extra)
    return [{"kb_idx": int(i), "folder": kb_folder[i], "chunk_id": kb_chunks[i]["chunk_id"],
             "source_file": kb_chunks[i]["source_file"], "score": round(float(score[i]), 4),
             "mapped": i in mset} for i in top], n_m, swaps

kb_retrieval, log = {}, []
for h in hadms:
    kb_retrieval[h] = {}
    for spec, conds in cmap.items():
        kb_retrieval[h][spec] = {}
        for code in conds:
            res, n_m, swaps = retrieve(QV[f"{h}|{code}"], pool[spec], mapped[spec][code])
            kb_retrieval[h][spec][code] = {"chunks": res, "mapped_before_floor": n_m, "floor_swaps": swaps}
            log.append((spec, code, n_m, swaps))

out = "kb_retrieval_test.json" if RUN_MODE == "test" else "kb_retrieval.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump({"top_k": TOP_K_KB, "floor": FLOOR, "query": "MedGemma-27B summary, BGE no prefix",
               "results": kb_retrieval}, f)

# ---- summary (counts only)
print(f"\nSaved {out}")
print(f"{'Specialist':<20}{'Code':<8}{'own-guideline before floor (mean)':>34}{'floor triggered':>17}")
agg = collections.defaultdict(list)
for spec, code, n_m, swaps in log:
    agg[(spec, code)].append((n_m, swaps))
for (spec, code), v in agg.items():
    print(f"{spec:<20}{code:<8}{np.mean([a for a, _ in v]):>34.1f}{sum(b > 0 for _, b in v):>12} / {len(v)}")

In [ ]:
import json, collections
t = json.load(open("kb_retrieval_test.json"))["results"]
case = next(iter(t.values()))
for spec, conds in case.items():
    print(f"\n=== {spec}")
    for code, r in conds.items():
        others = collections.Counter(c["source_file"][:45] for c in r["chunks"] if not c["mapped"])
        print(f"  {code}: " + "; ".join(f"{n}x {s}" for s, n in others.most_common(3)))

In [ ]:
import json, collections
t = json.load(open("kb_retrieval.json"))["results"]
for spec in ["Cardiology", "Pulmonology", "Nephrology", "Infectious Diseases"]:
    c, total = collections.Counter(), 0
    for case in t.values():
        for code, r in case[spec].items():
            for ch in r["chunks"]:
                if not ch["mapped"]:
                    c[ch["source_file"][:45]] += 1; total += 1
    print(f"\n=== {spec}: {total} open-slot chunks")
    for s, n in c.most_common(4):
        print(f"  {100*n/total:5.1f}%  {s}")

In [ ]:
import json, re, collections, numpy as np, pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

OUT_DIR = Path("evidence"); OUT_DIR.mkdir(exist_ok=True)
KB_DIR  = Path("Knowledge Base/Processed_v2")
FOLDERS = ["cardiology", "pulmonology", "nephrology", "infectious_diseases"]

cases  = pd.read_csv("cases_60.csv", dtype=str)
subj   = dict(zip(cases.hadm_id, cases.subject_id))
cmap   = json.load(open("condition_guideline_map.json"))
pr     = json.load(open("patient_retrieval.json"))["results"]
kr     = json.load(open("kb_retrieval.json"))["results"]
chunks = [json.loads(l) for l in open("all_patient_chunks_v2.jsonl", encoding="utf-8") if l.strip()]
assert set(pr) == set(kr) == set(cases.hadm_id), "case sets differ between files"

kb_src = {}
for folder in FOLDERS:
    for l in open(KB_DIR / f"{folder}_chunks.jsonl", encoding="utf-8"):
        if l.strip():
            c = json.loads(l)
            kb_src[f"{folder}|{c['chunk_id']}"] = c

tok = AutoTokenizer.from_pretrained("bbarn4/medgemma-27b-text-it-GPTQ")
ntok = lambda s: len(tok.encode(s, add_special_tokens=False))

def fmt_patient(rows):                      # record order, section header once per run
    parts, cur = [], None
    for r in rows:
        s = chunks[r]["section"]
        if s != cur:
            parts.append(f"\n{s}"); cur = s
        parts.append(chunks[r]["text"].strip())
    return "\n".join(parts).strip()

def fmt_kb(groups, kbc):                    # groups: [(header, [keys])]
    parts = []
    for header, keys in groups:
        parts.append(f"\n{header}")
        parts += [f"[{kbc[k]['source_file']}]\n{kbc[k]['text'].strip()}" for k in keys]
    return "\n\n".join(parts).strip()

KEY_SEC = {"Cardiology": "PATIENT ECG", "Pulmonology": "BLOOD GAS",
           "Nephrology": "PATIENT CHEMISTRY", "Infectious Diseases": "MICROBIOLOGY"}
cov_has, cov_hit = collections.Counter(), collections.Counter()
rows_by_hadm = collections.defaultdict(set)
for i, c in enumerate(chunks):
    rows_by_hadm[str(c["hadm_id"])].add(c["section"])

log = []
for hadm in cases.hadm_id:
    ev = {"hadm_id": hadm, "subject_id": subj[hadm],
          "patient_chunks": {}, "kb_chunks": {}, "specialists": {}}

    # patient chunks: stored once per case
    for code, lst in pr[hadm].items():
        for r in lst:
            pc = ev["patient_chunks"].setdefault(str(r["row"]), {
                "section": chunks[r["row"]]["section"], "chunk_id": chunks[r["row"]].get("chunk_id"),
                "text": chunks[r["row"]]["text"], "retrieved_for": [], "best_score": r["score"]})
            if code not in pc["retrieved_for"]:
                pc["retrieved_for"].append(code)
            pc["best_score"] = max(pc["best_score"], r["score"])

    all_p, spec_groups = set(), []
    for spec, conds in cmap.items():
        sp, p_rows, seen_kb, groups = {"conditions": {}}, set(), set(), []
        for code, meta in conds.items():
            rows = [r["row"] for r in pr[hadm][code]]
            p_rows.update(rows)
            kbl = []
            for c in kr[hadm][spec][code]["chunks"]:
                key = f"{c['folder']}|{c['chunk_id']}"
                src = kb_src[key]
                ev["kb_chunks"].setdefault(key, {"folder": c["folder"], "chunk_id": c["chunk_id"],
                                                 "source_file": src["source_file"], "text": src["text"]})
                kbl.append({"key": key, "score": c["score"], "mapped": c["mapped"]})
            kbl.sort(key=lambda x: -x["score"])
            sp["conditions"][code] = {"name": meta["name"], "patient_rows": rows, "kb": kbl,
                                      "kb_floor_swaps": kr[hadm][spec][code]["floor_swaps"]}
            new = [x["key"] for x in kbl if x["key"] not in seen_kb]
            seen_kb.update(new)
            groups.append((f"Candidate condition: {meta['name']} ({code})", new))
        sp["patient_rows"] = sorted(p_rows)
        sp["kb_groups"] = groups
        ev["specialists"][spec] = sp
        all_p |= p_rows
        spec_groups.append((spec, groups))

        # specialist-level coverage of its key section
        sec = KEY_SEC[spec]
        if sec in rows_by_hadm[hadm]:
            cov_has[spec] += 1
            cov_hit[spec] += any(chunks[r]["section"] == sec for r in p_rows)

        pt, kt = ntok(fmt_patient(sp["patient_rows"])), ntok(fmt_kb(groups, ev["kb_chunks"]))
        log.append({"hadm_id": hadm, "view": spec, "patient_chunks": len(p_rows),
                    "kb_chunks": len(seen_kb), "patient_tokens": pt, "kb_tokens": kt,
                    "total_tokens": pt + kt})

    # Model 2: everything, deduplicated across specialists
    seen, m2 = set(), []
    for spec, groups in spec_groups:
        for header, keys in groups:
            new = [k for k in keys if k not in seen]; seen.update(new)
            m2.append((f"{spec} | {header}", new))
    ev["model2"] = {"patient_rows": sorted(all_p), "kb_groups": m2}
    pt, kt = ntok(fmt_patient(ev["model2"]["patient_rows"])), ntok(fmt_kb(m2, ev["kb_chunks"]))
    log.append({"hadm_id": hadm, "view": "Model2", "patient_chunks": len(all_p),
                "kb_chunks": len(seen), "patient_tokens": pt, "kb_tokens": kt, "total_tokens": pt + kt})

    with open(OUT_DIR / f"{hadm}.json", "w", encoding="utf-8") as f:
        json.dump(ev, f)

df = pd.DataFrame(log)
df.to_csv("evidence_token_log.csv", index=False)

# ---- summary (counts only)
print(f"Evidence files written: {len(list(OUT_DIR.glob('*.json')))}")
print("\nTokens per view (median / max across 60 cases):")
for view, g in df.groupby("view", sort=False):
    print(f"  {view:<20} patient {int(g.patient_tokens.median()):>6,} / {g.patient_tokens.max():>6,}"
          f" | KB {int(g.kb_tokens.median()):>6,} / {g.kb_tokens.max():>6,}"
          f" | total {int(g.total_tokens.median()):>6,} / {g.total_tokens.max():>6,}"
          f" | chunks P {int(g.patient_chunks.median())} KB {int(g.kb_chunks.median())}")
print("\nSpecialist sees its key section (cases with it / cases that have the section):")
for spec, sec in KEY_SEC.items():
    print(f"  {spec:<20} {sec:<18} {cov_hit[spec]:>2} / {cov_has[spec]}")
summ = json.load(open("kb_query_summaries.json"))
print(f"\nSummaries containing digits: {sum(bool(re.search(r'\d', v['summary'])) for v in summ.values())} / {len(summ)}")